In [1]:
%%capture
%pip install -e ..

In [2]:
import os
import re
import yaml
import random
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from sklearn.metrics import cohen_kappa_score
from sycophancy.utils import lookup_answer
from sycophancy.verdict_analyser import VerdictAnalyser
from sycophancy.sycophancy_analyser import SycophancyAnalyser

In [3]:
load_dotenv()
MODELS = ["gpt-oss:120b", "qwen3:4b"]
RUNS = ["run_1", "run_2", "run_3"]
DATA_DIR = Path().cwd().parent / "data"

# Flip Rates

### Comparisons A and B

In [4]:
for model in MODELS:
    results_negative = []
    results_positive = []
    for run in RUNS:
        print(f"RESULTS FOR MODEL: {model}, RUN: {run}\n")
        with open(DATA_DIR / f"answers_{model}_full_{run}.yaml", "r") as file:
            outputs = yaml.safe_load(file)

        verdict_analyser = VerdictAnalyser(outputs)
        negative, positive = verdict_analyser.flip_rate_analysis(list(range(150))) # All questions
        results_negative.append(negative) 
        results_positive.append(positive)        
        print("\n---------------------")
        print("---------------------\n")


    print(f"OVERALL RESULTS FOR MODEL: {model}\n")
    print("COMPARISON A: OVERALL")
    same_output_negative = sum([result_negative["same_output"] for result_negative in results_negative]) / len(results_negative)
    different_output_negative = sum([result_negative["different_output"] for result_negative in results_negative]) / len(results_negative)
    expected_negative = sum([result_negative["expected"] for result_negative in results_negative]) / len(results_negative)
    unexpected_negative = sum([result_negative["unexpected"] for result_negative in results_negative]) / len(results_negative)
    overall_negative_results = {
        "same_output" : same_output_negative, 
        "different_output" : different_output_negative,
        "expected" : expected_negative, 
        "unexpected" : unexpected_negative,
    }
    verdict_analyser.print_flip_rate_result(overall_negative_results, memory = "negative")

    print("\n-----\n")
    print("COMPARISON B: OVERALL")
    same_output_positive = sum([result_positive["same_output"] for result_positive in results_positive]) / len(results_positive)
    different_output_positive = sum([result_positive["different_output"] for result_positive in results_positive]) / len(results_positive)
    expected_positive = sum([result_positive["expected"] for result_positive in results_positive]) / len(results_positive)
    unexpected_positive = sum([result_positive["unexpected"] for result_positive in results_positive]) / len(results_positive)
    overall_positive_results = {
        "same_output" : same_output_positive, 
        "different_output" : different_output_positive,
        "expected" : expected_positive, 
        "unexpected" : unexpected_positive,
    }
    verdict_analyser.print_flip_rate_result(overall_positive_results, memory = "positive")

    print("\n---------------------")
    print("---------------------\n")
    

RESULTS FOR MODEL: gpt-oss:120b, RUN: run_1

COMPARISON A: PER RUN
Same outputs - negative memory: 129.0
Output changed after adding memory - negative memory: 21.0
Expected flip - negative memory: 17.0
Unxpected flip - negative memory: 4.0

--------

Flip rate - negative memory: 14.0%
Proportion of expected flips - negative memory: 81.0%
Proportion of unexpected flips - negative memory: 19.0%


-----

COMPARISON B: PER RUN
Same outputs - positive memory: 134.0
Output changed after adding memory - positive memory: 16.0
Expected flip - positive memory: 7.0
Unxpected flip - positive memory: 9.0

--------

Flip rate - positive memory: 10.7%
Proportion of expected flips - positive memory: 43.8%
Proportion of unexpected flips - positive memory: 56.2%


---------------------
---------------------

RESULTS FOR MODEL: gpt-oss:120b, RUN: run_2

COMPARISON A: PER RUN
Same outputs - negative memory: 126.0
Output changed after adding memory - negative memory: 24.0
Expected flip - negative memory: 1

### Comparisons C and D

In [4]:
for model in MODELS:
    results_negative = []
    results_positive = []
    for run in RUNS:
        print(f"RESULTS FOR MODEL: {model}, RUN: {run}\n")
        with open(DATA_DIR / f"answers_{model}_full_{run}.yaml", "r") as file:
            outputs = yaml.safe_load(file)

        verdict_analyser = VerdictAnalyser(outputs)
        negative, positive = verdict_analyser.consensus_analysis() # All questions
        results_negative.append(negative) 
        results_positive.append(positive)
        
        print("\n---------------------")
        print("---------------------\n")

    print(f"OVERALL RESULTS FOR MODEL: {model}\n")
    print("COMPARISON C: OVERALL")
    no_flip_for_both_neg = sum([result_negative["no_flip_for_both"] for result_negative in results_negative]) / len(results_negative)
    only_flip_for_NTA_neg = sum([result_negative["only_flip_for_NTA"] for result_negative in results_negative]) / len(results_negative)
    only_flip_for_YTA_neg = sum([result_negative["only_flip_for_YTA"] for result_negative in results_negative]) / len(results_negative)
    flip_for_both_neg = sum([result_negative["flip_for_both"] for result_negative in results_negative]) / len(results_negative)
    expected_flips_NTA_neg = sum([result_negative["expected_flips_NTA"] for result_negative in results_negative]) / len(results_negative)
    expected_flips_YTA_neg = sum([result_negative["expected_flips_YTA"] for result_negative in results_negative]) / len(results_negative)
    unexpected_flips_NTA_neg = sum([result_negative["unexpected_flips_NTA"] for result_negative in results_negative]) / len(results_negative)
    unexpected_flips_YTA_neg = sum([result_negative["unexpected_flips_YTA"] for result_negative in results_negative]) / len(results_negative)

    overall_negative_results = {
        "no_flip_for_both" : no_flip_for_both_neg,
        "only_flip_for_NTA" : only_flip_for_NTA_neg,
        "only_flip_for_YTA" : only_flip_for_YTA_neg,
        "flip_for_both" : flip_for_both_neg,
        "expected_flips_NTA" : expected_flips_NTA_neg,
        "expected_flips_YTA" : expected_flips_YTA_neg,
        "unexpected_flips_NTA" : unexpected_flips_NTA_neg,
        "unexpected_flips_YTA" : unexpected_flips_YTA_neg,
    }
    verdict_analyser.print_consensus_analysis(overall_negative_results, memory = "negative")

    print("\n-----\n")
    print("COMPARISON D: OVERALL")
    no_flip_for_both_pos = sum([result_positive["no_flip_for_both"] for result_positive in results_positive]) / len(results_positive)
    only_flip_for_NTA_pos = sum([result_positive["only_flip_for_NTA"] for result_positive in results_positive]) / len(results_positive)
    only_flip_for_YTA_pos = sum([result_positive["only_flip_for_YTA"] for result_positive in results_positive]) / len(results_positive)
    flip_for_both_pos = sum([result_positive["flip_for_both"] for result_positive in results_positive]) / len(results_positive)
    expected_flips_NTA_pos = sum([result_positive["expected_flips_NTA"] for result_positive in results_positive]) / len(results_positive)
    expected_flips_YTA_pos = sum([result_positive["expected_flips_YTA"] for result_positive in results_positive]) / len(results_positive)
    unexpected_flips_NTA_pos = sum([result_positive["unexpected_flips_NTA"] for result_positive in results_positive]) / len(results_positive)
    unexpected_flips_YTA_pos = sum([result_positive["unexpected_flips_YTA"] for result_positive in results_positive]) / len(results_positive)

    overall_positive_results = {
        "no_flip_for_both" : no_flip_for_both_pos,
        "only_flip_for_NTA" : only_flip_for_NTA_pos,
        "only_flip_for_YTA" : only_flip_for_YTA_pos,
        "flip_for_both" : flip_for_both_pos,
        "expected_flips_NTA" : expected_flips_NTA_pos,
        "expected_flips_YTA" : expected_flips_YTA_pos,
        "unexpected_flips_NTA" : unexpected_flips_NTA_pos,
        "unexpected_flips_YTA" : unexpected_flips_YTA_pos,
    }
    verdict_analyser.print_consensus_analysis(overall_positive_results, memory = "positive")

    print("\n---------------------")
    print("---------------------\n")
    

RESULTS FOR MODEL: gpt-oss:120b, RUN: run_1

COMPARISON C: PER RUN
Both NTA and flipped NTA kept the same verdict given a negative self-description: 37 (74.0%)
Only NTA flipped verdict given a negative self-description: 6 (12.0%)
Only flipped NTA flipped verdict given a negative self-description: 3 (6.0%)
Both NTA and flipped NTA flipped verdict given a negative self-description: 4 (8.0%)
Expected flip of NTA given a negative self-description: 8.0
Expected flip of flipped NTA given a negative self-description: 5.0
Unexpected flip of NTA given a negative self-description: 2.0
Unexpected flip of flipped NTA given a negative self-description: 2.0

--------

COMPARISON D: PER RUN
Both NTA and flipped NTA kept the same verdict given a positive self-description: 41 (82.0%)
Only NTA flipped verdict given a positive self-description: 7 (14.0%)
Only flipped NTA flipped verdict given a positive self-description: 2 (4.0%)
Both NTA and flipped NTA flipped verdict given a positive self-description:

# Moral Sycophancy

In [5]:
for model in MODELS:
    neutral_results = []
    positive_results = []
    negative_results = []
    for run in RUNS:
        print(f"RESULTS FOR MODEL: {model}, RUN: {run}\n")
        with open(DATA_DIR / f"answers_{model}_full_{run}.yaml", "r") as file:
            outputs = yaml.safe_load(file)

        verdict_analyser = VerdictAnalyser(outputs)
        neutral, positive, negative = verdict_analyser.moral_sycophancy() 
        neutral_results.append(neutral)
        negative_results.append(negative)
        positive_results.append(positive)
        print("\n---------------------")
        print("---------------------\n")

    neutral_result = sum(neutral_results) / len(neutral_results)
    negative_result = sum(negative_results) / len(negative_results)
    positive_result = sum(positive_results) / len(positive_results)

    print(f"OVERALL RESULTS FOR MODEL: {model}\n")
    print(f"Morality score for neutral self-description: {neutral_result:.2f}%")
    print(f"Morality score for positive self-description: {positive_result:.2f}%")
    print(f"Morality score for negative self-description: {negative_result:.2f}%")

    print("\n---------------------")
    print("---------------------\n")
    
    
    

RESULTS FOR MODEL: gpt-oss:120b, RUN: run_1

Morality score for neutral self-description: 42.00%
Morality score for positive self-description: 38.00%
Morality score for negative self-description: 24.00%


---------------------
---------------------

RESULTS FOR MODEL: gpt-oss:120b, RUN: run_2

Morality score for neutral self-description: 42.00%
Morality score for positive self-description: 42.00%
Morality score for negative self-description: 32.00%


---------------------
---------------------

RESULTS FOR MODEL: gpt-oss:120b, RUN: run_3

Morality score for neutral self-description: 36.00%
Morality score for positive self-description: 36.00%
Morality score for negative self-description: 30.00%


---------------------
---------------------

OVERALL RESULTS FOR MODEL: gpt-oss:120b

Morality score for neutral self-description: 40.00%
Morality score for positive self-description: 38.67%
Morality score for negative self-description: 28.67%

---------------------
---------------------

RESUL

In [ ]:
for model in MODELS:
    for run in RUNS:
        print(f"RESULTS FOR MODEL: {model}, RUN: {run}\n")
        with open(DATA_DIR / f"answers_{model}_full_{run}.yaml", "r") as file:
            outputs = yaml.safe_load(file)

        verdict_analyser = VerdictAnalyser(outputs)
        verdict_analyser.moral_sycophancy_extended() 
        print("\n---------------------")
        print("---------------------\n")

RESULTS FOR MODEL: gpt-oss:120b, RUN: run_1

Original (/|P|)
	Only looking at NTA:
		Morality score for neutral self-description: 42.00%
		Morality score for positive self-description: 38.00%
		Morality score for negative self-description: 24.00%

	Only looking at YTA:
		Morality score for neutral self-description: 4.00%
		Morality score for positive self-description: 6.00%
		Morality score for negative self-description: 4.00%

Adjusted (/number of original NTA or YTA)
	Only looking at NTA:
		Morality score for neutral self-description: 50.00%
		Morality score for positive self-description: 48.72%
		Morality score for negative self-description: 33.33%

	Only looking at YTA:
		Morality score for neutral self-description: 4.76%
		Morality score for positive self-description: 7.69%
		Morality score for negative self-description: 5.56%

Overall morality score:
Morality score for neutral self-description: 46.00%
Morality score for positive self-description: 44.00%
Morality score for negativ

# Validation and Framing Sycophancy

## Validation

In [4]:
for model in MODELS:
    results = []
    for run in RUNS:
        print(f"RESULTS FOR MODEL: {model}, RUN: {run}\n")
        with open(DATA_DIR / "annotation" / f"results_{model}_full_{run}.csv", "r") as file:
            all_results_final_df = pd.read_csv(file)

        sycophancy_analyser = SycophancyAnalyser(all_results_final_df)
        result = sycophancy_analyser.validation_sycophancy()
        results.append(result)
        print("\n---------------------")
        print("---------------------\n")

    binary_neutral_validation = sum([result["binary_neutral_validation"] for result in results]) / len(results)
    binary_positive_validation = sum([result["binary_positive_validation"] for result in results]) / len(results)
    binary_negative_validation = sum([result["binary_negative_validation"] for result in results]) / len(results)

    likert_neutral_validation = sum([result["likert_neutral_validation"] for result in results]) / len(results)
    likert_positive_validation = sum([result["likert_positive_validation"] for result in results]) / len(results)
    likert_negative_validation = sum([result["likert_negative_validation"] for result in results]) / len(results)

    print(f"OVERALL RESULTS FOR MODEL: {model}\n")
    print(f"Binary validation with neutral self-description: {binary_neutral_validation:.2f}")
    print(f"Binary validation with positive self-description: {binary_positive_validation:.2f}")
    print(f"Binary validation with negative self-description: {binary_negative_validation:.2f}")

    print(f"Likert validation with neutral self-description: {likert_neutral_validation:.2f}")
    print(f"Likert validation with positive self-description: {likert_positive_validation:.2f}")
    print(f"Likert validation with negative self-description: {likert_negative_validation:.2f}")

    print("\n---------------------")
    print("---------------------\n")

RESULTS FOR MODEL: gpt-oss:120b, RUN: run_1

Binary validation with neutral self-description: 0.72
Binary validation with positive self-description: 0.70
Binary validation with negative self-description: 0.61
Likert validation with neutral self-description: 3.56
Likert validation with positive self-description: 3.50
Likert validation with negative self-description: 3.26

---------------------
---------------------

RESULTS FOR MODEL: gpt-oss:120b, RUN: run_2

Binary validation with neutral self-description: 0.72
Binary validation with positive self-description: 0.68
Binary validation with negative self-description: 0.61
Likert validation with neutral self-description: 3.59
Likert validation with positive self-description: 3.53
Likert validation with negative self-description: 3.22

---------------------
---------------------

RESULTS FOR MODEL: gpt-oss:120b, RUN: run_3

Binary validation with neutral self-description: 0.64
Binary validation with positive self-description: 0.66
Binary v

## Framing

In [5]:
for model in MODELS:
    results = []
    for run in RUNS:
        print(f"RESULTS FOR MODEL: {model}, RUN: {run}\n")
        with open(DATA_DIR / "annotation" / f"results_{model}_full_{run}.csv", "r") as file:
            all_results_final_df = pd.read_csv(file)

        sycophancy_analyser = SycophancyAnalyser(all_results_final_df)
        result = sycophancy_analyser.framing_sycophancy()
        results.append(result)
        print("\n---------------------")
        print("---------------------\n")

    binary_neutral_framing = sum([result["binary_neutral_framing"] for result in results]) / len(results)
    binary_positive_framing = sum([result["binary_positive_framing"] for result in results]) / len(results)
    binary_negative_framing = sum([result["binary_negative_framing"] for result in results]) / len(results)

    likert_neutral_framing = sum([result["likert_neutral_framing"] for result in results]) / len(results)
    likert_positive_framing = sum([result["likert_positive_framing"] for result in results]) / len(results)
    likert_negative_framing = sum([result["likert_negative_framing"] for result in results]) / len(results)

    print(f"OVERALL RESULTS FOR MODEL: {model}\n")
    print(f"Binary framing with neutral self-description: {binary_neutral_framing:.2f}")
    print(f"Binary framing with positive self-description: {binary_positive_framing:.2f}")
    print(f"Binary framing with negative self-description: {binary_negative_framing:.2f}")

    print(f"Likert framing with neutral self-description: {likert_neutral_framing:.2f}")
    print(f"Likert framing with positive self-description: {likert_positive_framing:.2f}")
    print(f"Likert framing with negative self-description: {likert_negative_framing:.2f}")

    print("\n---------------------")
    print("---------------------\n")

RESULTS FOR MODEL: gpt-oss:120b, RUN: run_1

Binary framing with neutral self-description: 0.81
Binary framing with positive self-description: 0.76
Binary framing with negative self-description: 0.74
Likert framing with neutral self-description: 3.99
Likert framing with positive self-description: 3.76
Likert framing with negative self-description: 3.75

---------------------
---------------------

RESULTS FOR MODEL: gpt-oss:120b, RUN: run_2

Binary framing with neutral self-description: 0.83
Binary framing with positive self-description: 0.82
Binary framing with negative self-description: 0.75
Likert framing with neutral self-description: 4.08
Likert framing with positive self-description: 4.01
Likert framing with negative self-description: 3.81

---------------------
---------------------

RESULTS FOR MODEL: gpt-oss:120b, RUN: run_3

Binary framing with neutral self-description: 0.77
Binary framing with positive self-description: 0.78
Binary framing with negative self-description: 0.7